<a href="https://colab.research.google.com/github/sxsaa/Mathematical-Reasoning-in-Small-Language-Models-using-Process-Reward-Models-and-Best-of-N-Search/blob/main/Application_with_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers accelerate bitsandbytes gradio

In [ ]:
import math
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
import gradio as gr
import re

# ==========================================
# 1. INITIALIZE & LOAD MODELS (Runs once)
# ==========================================
print("Loading Generator Model (1.5B Math)...")
gen_id = "Qwen/Qwen2.5-Math-1.5B-Instruct"
gen_tokenizer = AutoTokenizer.from_pretrained(gen_id)
gen_model = AutoModelForCausalLM.from_pretrained(
    gen_id,
    torch_dtype=torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16,
    device_map="auto"
).eval()

print("Loading Verifier Model (3B PRM)...")
ver_id = "sxsaa/Qwen2.5-3B-Math-Verifier-FullData-v2.0"
ver_tokenizer = AutoTokenizer.from_pretrained(ver_id)
ver_model = AutoModelForCausalLM.from_pretrained(
    ver_id,
    torch_dtype=torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16,
    device_map="auto"
).eval()

correct_id = ver_tokenizer.encode("Correct", add_special_tokens=False)[0]
neutral_id = ver_tokenizer.encode("Neutral", add_special_tokens=False)[0]
incorrect_id = ver_tokenizer.encode("Incorrect", add_special_tokens=False)[0]

# ==========================================
# 2. CORE LOGIC & UI FORMATTING
# ==========================================
def analyze_math_problem(problem, progress=gr.Progress()):
    progress(0, desc="Generating 3 solutions...")

    few_shot_prompt = f"""You are a logical math assistant. Solve the math problem step-by-step.
ABSOLUTE RULES:
1. NO NUMBERING: NEVER start a line with numbers like "1.", "2.", "Step 1", etc.
2. NO DANGLING INTROS: NEVER write a phrase that just introduces the next line. You MUST put the math on the exact SAME line as the explanation.
3. NO COLONS: NEVER use a colon (:).
4. NO FORMATTING: Do NOT use bullet points, bold text, or LaTeX symbols (like \\ldots, \\pmod, \\equiv).
5. HUMAN READABLE: Use plain text for complex math. Write "10 choose 2" instead of \\binom{{10}}{{2}}. Write "a/b" instead of \\frac{{a}}{{b}}. Write "sqrt(x)" instead of \\sqrt{{x}}.

Example 1:
Problem: Solve 2x + 6 = 14
We subtract 6 from both sides to get 2x = 8.
We divide both sides by 2 to get x = 4.
The final answer is 4.

Example 2:
Problem: Find the largest two-digit integer leaving a remainder of 2 when divided by 8.
Let the number be 8k + 2.
Since it is a two-digit number we know that 8k + 2 <= 99.
We subtract 2 from both sides to get 8k <= 97.
We divide by 8 to get k <= 12.125.
The largest integer k is therefore 12.
We substitute k back in to get 8(12) + 2 = 98.
The final answer is 98.

Problem: {problem}"""

    messages = [
        {"role": "system", "content": "You are a strict math assistant that writes plain-text, single-sentence steps without using colons, numbering, or line breaks before equations."},
        {"role": "user", "content": few_shot_prompt}
    ]

    text = gen_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = gen_tokenizer([text], return_tensors="pt").to(gen_model.device)

    with torch.no_grad():
        outputs = gen_model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            num_return_sequences=3,
            pad_token_id=gen_tokenizer.eos_token_id
        )

    generated_texts = gen_tokenizer.batch_decode(outputs[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)

    ranked_results = []
    html_output = f"<h2>Problem: {problem}</h2><hr>"

    for i, solution_text in enumerate(generated_texts):
        progress((i + 1) / 3, desc=f"Verifying Solution {i + 1}...")

        # --- THE REFINED REGEX CLEANUP ---

        # 1. Strip ALL markdown bolding and LaTeX formatting FIRST
        replacements = {
            '**': '', '\\[': '', '\\]': '', '\\(': '', '\\)': '', '$$': '',
            '\\leq': '<=', '\\geq': '>=', '\\cdot': '*', '\\times': '*', '\\approx': '~',
            '\\ldots': '...', '\\equiv': '==', '\\pmod': 'mod'
        }
        for old, new in replacements.items():
            solution_text = solution_text.replace(old, new)

        # 2. Strip starting numbers from any line
        solution_text = re.sub(r'(?m)^\s*(?:Step\s*\d+[:.]?|\d+[\.)])\s*', '', solution_text)

        # 3. Glue dangling phrases ending in "get", "are", or "is" to the FIRST equation below it
        solution_text = re.sub(r'(:|get|are|is|have|yields|form|inequality|equation|follows|by)\s*\n\s*', r'\1 ', solution_text)

        # 4. Vaporize colons
        solution_text = solution_text.replace(':', '')

        # ---> FIX: Clean up System of Equations <---
        solution_text = solution_text.replace('\\begin{cases}', '')
        solution_text = solution_text.replace('\\end{cases}', '')
        solution_text = solution_text.replace('\\\\', '\n') # Translates LaTeX newlines to actual newlines

        # 5. Handle complex LaTeX translations
        solution_text = re.sub(r'\\binom\{([^}]+)\}\{([^}]+)\}', r'\1 choose \2', solution_text)
        solution_text = re.sub(r'\\boxed\{([^}]+)\}', r'\1', solution_text)
        solution_text = re.sub(r'\\frac\{([^}]+)\}\{([^}]+)\}', r'\1/\2', solution_text)

        # 6. Beautiful Square Roots and Powers
        solution_text = re.sub(r'\\sqrt\{([^}]+)\}', r'√(\1)', solution_text)
        solution_text = solution_text.replace('\\sqrt', '√')
        solution_text = solution_text.replace('^2', '²').replace('^3', '³')
        # ------------------------------------------

        steps = [s.strip() for s in solution_text.split('\n') if s.strip()]
        if not steps:
            continue

        history = []
        cumulative_log_score = 0.0

        solution_html = f"<h3>Solution {i+1}</h3><div style='background-color: #f9f9f9; padding: 15px; border-radius: 8px; margin-bottom: 5px; font-family: monospace;'>"

        for step in steps:
            history_text = "\n".join(history) if history else "No previous steps."

            ver_prompt = (
                f"Problem: {problem}\n\n"
                f"History of Previous Steps:\n{history_text}\n\n"
                f"Current Step to Evaluate: {step}\n\n"
                f"Question: Label this step as Correct, Neutral, or Incorrect."
            )
            ver_messages = [{"role": "user", "content": ver_prompt}]
            ver_text = ver_tokenizer.apply_chat_template(ver_messages, tokenize=False, add_generation_prompt=True)
            ver_inputs = ver_tokenizer([ver_text], return_tensors="pt").to(ver_model.device)

            with torch.no_grad():
                logits = ver_model(**ver_inputs).logits[:, -1, :]

            probs = F.softmax(logits[0, [correct_id, neutral_id, incorrect_id]], dim=-1)

            p_correct = probs[0].item()
            best_idx = torch.argmax(probs).item()
            labels_list = ["Correct", "Neutral", "Incorrect"]
            predicted_label = labels_list[best_idx]

            safe_prob = max(p_correct, 1e-10)
            cumulative_log_score += math.log(safe_prob)

            if predicted_label == "Correct":
                color = "green"
            elif predicted_label == "Incorrect":
                color = "red"
            else:
                color = "orange"

            solution_html += f"<span style='color: {color}; display: block; margin-bottom: 5px;'>{step}</span>"
            history.append(step)

        N = len(steps)
        final_score = cumulative_log_score / N

        solution_html += "</div>"
        solution_html += f"<p style='margin-bottom: 20px;'><b>Calculated Length-Normalized Score: {final_score:.4f}</b></p>"

        ranked_results.append({
            "solution_id": i + 1,
            "html": solution_html,
            "score": final_score,
            "raw_text": solution_text
        })

    ranked_results.sort(key=lambda x: x['score'], reverse=True)
    winner = ranked_results[0]

    for res in ranked_results:
        html_output += res["html"]

    html_output += "<hr><div style='background-color: #e6ffe6; padding: 20px; border-left: 5px solid green; border-radius: 5px; margin-top: 30px;'>"
    html_output += f"<h2>🏆 Best Solution: {winner['solution_id']}</h2>"
    html_output += f"<p><b>Score: {winner['score']:.4f}</b></p><br>"

    winning_steps = [s.strip() for s in winner['raw_text'].split('\n') if s.strip()]
    for step in winning_steps:
         html_output += f"<span style='display: block; margin-bottom: 5px;'>{step}</span>"

    html_output += "</div>"

    return html_output

# ==========================================
# 3. LAUNCH THE GRADIO UI
# ==========================================
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🧮 Process Reward Model: Best-of-3 Math Solver")
    gr.Markdown("Enter a math problem below. The **1.5B Math Model** will brainstorm 3 solutions, and fine-tuned 3B Verifier will grade them step-by-step. <br><span style='color:green; font-weight:bold;'>Green = Correct</span> | <span style='color:orange; font-weight:bold;'>Orange = Neutral</span> | <span style='color:red; font-weight:bold;'>Red = Incorrect</span>.")

    with gr.Row():
        input_box = gr.Textbox(lines=3, placeholder="Enter a math problem here...", label="Math Problem")

    submit_btn = gr.Button("Generate & Evaluate Solutions", variant="primary")

    output_html = gr.HTML(label="Evaluation Results")

    submit_btn.click(fn=analyze_math_problem, inputs=input_box, outputs=output_html)

demo.launch(debug=True)